# 🏋️ Fine-Tuning de BioMistral-7B com MedQuAD

**Tech Challenge FIAP - Fase 3 | Assistente Médico Inteligente**

⚠️ **VERSÃO SEGURA**: Este notebook salva TUDO no Google Drive desde o início.
Se a sessão cair, basta reabrir e retomar com `resume_from_checkpoint=True`.

---

## 🎯 O que este notebook faz

Pega o modelo **BioMistral-7B** (já pré-treinado em PubMed) e ajusta ele especificamente com perguntas/respostas médicas do dataset **MedQuAD anonimizado**.

## ⏱️ Tempo estimado

- Setup: ~5 min
- Download modelo: ~10 min
- Treinamento: ~2-4h (2 epochs, A100)
- Salvar no Drive: ~1 min
- **TOTAL: ~3-5h** (pode deixar rodando sem supervisão após os primeiros 10 min)

## 🛡️ Segurança

- ✅ Tudo salvo em `/content/drive/MyDrive/techchallenge_fase3/`
- ✅ Checkpoints a cada epoch no Drive
- ✅ Retomável se cair (via `resume_from_checkpoint`)

---

# SEÇÃO 1: Setup do ambiente + Verificação GPU

**Antes de começar**:
1. Runtime → Change runtime type → Hardware: **A100 GPU**
2. Confirme que está rodando esta célula abaixo

In [ ]:
# ============================================================
# SEÇÃO 1: Setup + Verificação GPU
# ============================================================
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print("=" * 60)
print(f"🖥️  GPU: {torch.cuda.get_device_name(0)}")
print(f"🖥️  VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"🐍  Python: {torch.__version__}")
print("=" * 60)

if torch.cuda.get_device_properties(0).total_mem / 1e9 < 30:
    print("⚠️  AVISO: GPU tem menos de 30GB. Recomendado A100 (40GB).")
    print("   Se cair em OOM, reduza per_device_train_batch_size pra 1")
else:
    print("✅ GPU perfeita pra BioMistral-7B + QLoRA!")

**Esperado**:
```
🖥️  GPU: NVIDIA A100-SXM4-40GB
🖥️  VRAM: 40.0 GB
✅ GPU perfeita pra BioMistral-7B + QLoRA!
```

❌ Se aparecer GPU diferente ou erro: volte em **Runtime → Change runtime type**

---

# SEÇÃO 2: Instalar dependências

In [ ]:
# ============================================================
# SEÇÃO 2: Instalar dependências (~2 min)
# ============================================================
%%capture  # esconde output verboso

!pip install -q torch==2.1.0+cu121 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.44.0 datasets==2.21.0 peft==0.10.0
!pip install -q bitsandbytes==0.43.3 accelerate==0.33.0 trl==0.10.0
!pip install -q unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git
!pip install -q loguru==0.7.2

print("✅ Todas as dependências instaladas!")

---

# SEÇÃO 3: Montar Google Drive (PERSISTÊNCIA)

⚠️ **MUITO IMPORTANTE**: vamos trabalhar **dentro do Drive** desde o início.
Isso garante que checkpoints e modelo final sobrevivam se a sessão cair.

In [ ]:
# ============================================================
# SEÇÃO 3: Montar Drive + configurar paths persistentes
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# Diretório de trabalho no Drive (PERSISTE mesmo se sessão cair)
WORKDIR = '/content/drive/MyDrive/techchallenge_fase3'
import os
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)

# Subdiretórios
CKPT_DIR = f"{WORKDIR}/checkpoints"          # checkpoints intermediários
MODEL_DIR = f"{WORKDIR}/biomistral-medquad-lora"  # modelo final
DATA_DIR = f"{WORKDIR}/data"                # onde vai o train.jsonl

for d in [CKPT_DIR, MODEL_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"✅ Working directory: {WORKDIR}")
print(f"   Checkpoints:  {CKPT_DIR}")
print(f"   Modelo final: {MODEL_DIR}")
print(f"   Dados:        {DATA_DIR}")
print()
print(f"📂 Conteúdo atual de {WORKDIR}:")
!ls -lh {WORKDIR}

## ⚠️ Agora faça upload do `train.jsonl` para o Drive

**Como fazer** (escolha uma):

### Opção A: Via PC → Drive (recomendado)
1. No seu PC, copie `C:\Users\Teste\Downloads\Techchalleng3\data\processed\train.jsonl` (17 MB)
2. Cole em `G:\Meu Drive\techchallenge_fase3\data\train.jsonl` (no seu Drive local)
3. Drive sincroniza automaticamente

### Opção B: Upload direto no Colab
```python
from google.colab import files
uploaded = files.upload()  # vai pedir pra selecionar arquivo
import shutil
shutil.move(list(uploaded.keys())[0], f"{DATA_DIR}/train.jsonl")
```

Depois de upar, **rode a célula abaixo** pra confirmar:

In [ ]:
# ============================================================
# SEÇÃO 3.1: Verificar que train.jsonl está no Drive
# ============================================================
import os

train_path = f"{DATA_DIR}/train.jsonl"
if os.path.exists(train_path):
    size_mb = os.path.getsize(train_path) / 1024 / 1024
    with open(train_path) as f:
        n_lines = sum(1 for _ in f)
    print(f"✅ train.jsonl encontrado!")
    print(f"   Tamanho: {size_mb:.1f} MB")
    print(f"   Amostras: {n_lines:,}")
else:
    print(f"❌ train.jsonl NÃO encontrado em {train_path}")
    print(f"   Faça upload antes de continuar (veja célula acima)")
    raise FileNotFoundError("Upload train.jsonl primeiro!")

---

# SEÇÃO 4: Carregar BioMistral-7B

Aqui é onde a mágica acontece. Vamos baixar o BioMistral-7B (~14GB) e configurar com QLoRA.

**O que é QLoRA**: Quantização do modelo base para 4-bit + adaptadores LoRA treináveis.
Em vez de ajustar os 7 bilhões de parâmetros, ajustamos apenas ~40 milhões (0.6% do total).

⏱️ **Esta célula leva ~5-10min na primeira vez** (download do modelo)

In [ ]:
# ============================================================
# SEÇÃO 4: Carregar BioMistral-7B com QLoRA
# ============================================================
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 4096  # tokens máximos por exemplo (instruction + input + output)
DTYPE = None           # auto-detectar (float16 ou bfloat16)
LOAD_IN_4BIT = True    # QLoRA: quantizar pra 4-bit (economiza 4x VRAM)

print("📥 Baixando BioMistral-7B... (pode levar 5-10 min na primeira vez)")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="BioMistral/BioMistral-7B",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    # token="hf_XXXXXXXXXX",  # descomente se precisar autenticar no HuggingFace
)

print("\n✅ BioMistral-7B carregado!")
print(f"   Vocabulário: {tokenizer.vocab_size:,} tokens")

---

# SEÇÃO 5: Configurar LoRA adapters

Agora vamos adicionar os 'notas adesivas' (LoRA) ao modelo.

In [ ]:
# ============================================================
# SEÇÃO 5: Configurar LoRA adapters
# ============================================================
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # rank LoRA (compromisso performance/velocidade)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,  # fator de escala (= 2 * rank)
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = model.num_parameters()
pct = 100 * trainable_params / all_params

print("=" * 60)
print("📊 PARÂMETROS DO MODELO")
print("=" * 60)
print(f"  Total:           {all_params:,} ({all_params/1e9:.2f}B)")
print(f"  Treináveis:      {trainable_params:,} ({trainable_params/1e6:.1f}M)")
print(f"  % Treinável:     {pct:.4f}%")
print()
print("💡 Apenas ~0.6% dos parâmetros são treinados!")
print("   Por isso LoRA é tão rápido e leve.")

---

# SEÇÃO 6: Carregar e formatar dataset

In [ ]:
# ============================================================
# SEÇÃO 6: Carregar dataset + template Alpaca
# ============================================================
from datasets import load_dataset

dataset = load_dataset("json", data_files=f"{DATA_DIR}/train.jsonl", split="train")
print(f"✅ Dataset carregado: {len(dataset):,} amostras")
print()
print(f"📝 Exemplo (amostra 0):")
print(f"   Instruction: {dataset[0]['instruction']}")
print(f"   Input:       {dataset[0]['input']}")
print(f"   Output:      {dataset[0]['output'][:200]}...")

In [ ]:
# ============================================================
# SEÇÃO 6.1: Aplicar template Alpaca
# ============================================================
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token


def format_example(example):
    """Formata 1 exemplo no template Alpaca."""
    text = alpaca_prompt.format(
        example["instruction"],
        example.get("input", ""),
        example["output"]
    ) + EOS_TOKEN
    return {"text": text}


dataset = dataset.map(format_example)

print("=" * 60)
print("✅ Dataset formatado no template Alpaca")
print("=" * 60)
print()
print(f"📝 Exemplo formatado (amostra 0):\n")
print(dataset[0]["text"][:600])
print("...")

---

# SEÇÃO 7: Configurar o Trainer

In [ ]:
# ============================================================
# SEÇÃO 7: Configurar SFTTrainer
# ============================================================
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,         # batch efetivo = 8
        warmup_steps=50,
        num_train_epochs=2,                   # 2 epochs é o sweet spot
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=20,                      # log a cada 20 steps
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir=CKPT_DIR,                   # ← SALVA NO DRIVE!
        save_strategy="epoch",                # salva após cada epoch
        save_total_limit=2,                   # mantém 2 últimos
        report_to="none",                     # sem wandb
    ),
)

print("✅ Trainer configurado!")
print(f"   Dataset:      {len(dataset):,} amostras")
print(f"   Batch efetivo: 2 × 4 = 8")
print(f"   Steps/epoch:  {len(dataset)//8:,}")
print(f"   Total steps:  {len(dataset)//8*2:,} (2 epochs)")
print(f"   Checkpoints:  {CKPT_DIR}")
print(f"   Tempo estimado A100: 2-4h")

---

# SEÇÃO 8: 🚀 TREINAR! (essa é a parte demorada)

Agora vamos **rodar o fine-tuning**. A célula abaixo pode levar 2-4h.

**Dicas:**
- ✅ Pode **minimizar** a janela — o treino continua
- ✅ Pode **fechar** o navegador — o treino continua!
- ✅ Verá mensagens de progresso a cada 20 steps
- ✅ Se cair, basta reabrir e usar `resume_from_checkpoint=True`

**⚠️ Importante**: Se você quer **retomar de onde parou** (caso tenha caído antes), mude a última linha para:
```python
trainer.train(resume_from_checkpoint=True)
```

In [ ]:
# ============================================================
# SEÇÃO 8: TREINAMENTO 🚀
# ============================================================
print("=" * 60)
print("🚀 INICIANDO TREINAMENTO")
print("=" * 60)
print("⏱️  Tempo estimado: 2-4 horas em A100")
print("💾 Checkpoints salvos em: " + CKPT_DIR)
print("💡 Pode fechar o navegador — o treino continua!")
print()

# ⚠️ Se você REINICIOU após queda, descomente a linha abaixo:
# trainer.train(resume_from_checkpoint=True)

trainer.train()

print()
print("=" * 60)
print("✅ TREINAMENTO CONCLUÍDO!")
print("=" * 60)

---

# SEÇÃO 9: Salvar modelo final no Drive

Após o treino, salvamos os adaptadores LoRA (são só ~80MB) no Drive.

In [ ]:
# ============================================================
# SEÇÃO 9: Salvar modelo final no Drive
# ============================================================
model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

print("=" * 60)
print("💾 MODELO SALVO!")
print("=" * 60)
print(f"📂 Diretório: {MODEL_DIR}")
print()
!ls -lh {MODEL_DIR}/

import os
size_mb = sum(
    os.path.getsize(os.path.join(MODEL_DIR, f))
    for f in os.listdir(MODEL_DIR)
) / 1024 / 1024

print(f"\n📦 Tamanho total: {size_mb:.1f} MB")
print("   (Bem menor que o modelo completo de 14GB!)")

---

# SEÇÃO 10: Testar o modelo com perguntas reais

In [ ]:
# ============================================================
# SEÇÃO 10: Inferência — testar o modelo
# ============================================================
from transformers import TextStreamer

# Ativar modo inferência do Unsloth (mais rápido)
FastLanguageModel.for_inference(model)

def perguntar(instruction: str, topic: str = ""):
    """Faz uma pergunta ao modelo fine-tuned."""
    input_text = alpaca_prompt.format(
        instruction,
        f"Context / Topic: {topic}" if topic else "",
        ""
    )

    inputs = tokenizer([input_text], return_tensors="pt").to("cuda")
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    print(f"\n{'='*70}")
    print(f"❓ PERGUNTA: {instruction}")
    if topic:
        print(f"📋 TÓPICO: {topic}")
    print(f"{'='*70}")
    print(f"🤖 RESPOSTA:")

    _ = model.generate(
        **inputs,
        streamer=streamer,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.2,
    )
    print()

# Testar com 3 perguntas
perguntar(
    "What are the symptoms of diabetes type 2?",
    "Diabetes Type 2"
)

perguntar(
    "What are the treatments for high blood pressure?",
    "Hypertension"
)

perguntar(
    "Is breast cancer hereditary?",
    "Breast Cancer"
)

---

# SEÇÃO 11: Avaliação quantitativa (perplexity no val set)

In [ ]:
# ============================================================
# SEÇÃO 11: Avaliação — perplexity no val set
# ============================================================
import math
from datasets import load_dataset

# Carregar val.jsonl do Drive (deve estar lá se você upou train.jsonl completo)
VAL_FILE = f"{DATA_DIR}/val.jsonl"

if not os.path.exists(VAL_FILE):
    print(f"⚠️  val.jsonl não encontrado em {VAL_FILE}")
    print(f"   Se você só upou train.jsonl, copie val.jsonl também do seu PC.")
    print(f"   Val.jsonl está em: C:\\Users\\Teste\\Downloads\\Techchalleng3\\data\\processed\\val.jsonl")
else:
    val_dataset = load_dataset("json", data_files=VAL_FILE, split="val")
    print(f"📂 Carregado val set: {len(val_dataset):,} amostras\n")

    total_loss = 0
    n_samples = 0
    MAX_EVAL_SAMPLES = 100

    model.eval()
    with torch.no_grad():
        for i, example in enumerate(val_dataset):
            if i >= MAX_EVAL_SAMPLES:
                break

            text = alpaca_prompt.format(
                example["instruction"],
                example.get("input", ""),
                example["output"]
            )
            inputs = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_SEQ_LENGTH,
            ).to("cuda")

            outputs = model(**inputs, labels=inputs["input_ids"])
            total_loss += outputs.loss.item()
            n_samples += 1

            if (i + 1) % 20 == 0:
                print(f"   Avaliado {i+1}/{MAX_EVAL_SAMPLES} amostras...")

    perplexity = math.exp(total_loss / n_samples)

    print()
    print("=" * 60)
    print("📊 RESULTADO DA AVALIAÇÃO")
    print("=" * 60)
    print(f"  Loss média:    {total_loss / n_samples:.4f}")
    print(f"  Perplexity:    {perplexity:.2f}")
    print()
    print("💡 Interpretação:")
    print("   • < 5   = Modelo 'decorou' o val set")
    print("   • 5-15 = Excelente (aprendeu o domínio)")
    print("   • 15-30 = Bom")
    print("   • > 50 = Modelo ainda não aprendeu")

---

# SEÇÃO 12: Avaliação qualitativa — gerar respostas para revisão

In [ ]:
# ============================================================
# SEÇÃO 12: Avaliação qualitativa — comparar com gabarito
# ============================================================
import json
import os

TEST_FILE = f"{DATA_DIR}/test.jsonl"
EVAL_OUTPUT = f"{WORKDIR}/eval_results_qualitativo.json"

if not os.path.exists(TEST_FILE):
    print(f"⚠️  test.jsonl não encontrado. Pulando esta seção.")
else:
    test_dataset = load_dataset("json", data_files=TEST_FILE, split="test")

    # Pegar 20 amostras aleatórias
    import random
    rng = random.Random(42)
    indices = rng.sample(range(len(test_dataset)), min(20, len(test_dataset)))
    samples = [test_dataset[i] for i in indices]

    results = []
    print(f"🚀 Gerando respostas para {len(samples)} perguntas...\n")

    for i, example in enumerate(samples):
        input_text = alpaca_prompt.format(
            example["instruction"],
            example.get("input", ""),
            ""
        )
        inputs = tokenizer([input_text], return_tensors="pt").to("cuda")

        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.2,
        )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        response = response.split("### Response:")[-1].strip()

        results.append({
            "instruction": example["instruction"],
            "expected": example["output"][:500],
            "generated": response[:500],
        })

        print(f"  [{i+1}/20] {example['instruction'][:60]}... OK")

    # Salvar
    with open(EVAL_OUTPUT, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"\n✅ Resultados salvos em: {EVAL_OUTPUT}")
    print(f"   {len(results)} pares (esperado vs gerado) para revisão manual")

    # Mostrar 2 exemplos lado a lado
    print("\n" + "=" * 70)
    print("📋 EXEMPLOS LADO A LADO")
    print("=" * 70)
    for r in results[:2]:
        print(f"\n--- PERGUNTA ---")
        print(f"   {r['instruction']}")
        print(f"\n--- ESPERADO ---")
        print(f"   {r['expected'][:300]}...")
        print(f"\n--- GERADO ---")
        print(f"   {r['generated'][:300]}...")
        print()

---

# 🎉 CONCLUSÃO

Parabés! Você completou o **fine-tuning do BioMistral-7B** com o dataset MedQuAD.

## 📁 O que você tem agora

1. **Modelo fine-tuned**: `biomistral-medquad-lora/` (~80MB) — pronto pra usar no pipeline LangChain
2. **Resultados qualitativos**: `eval_results_qualitativo.json` — 20 pares pra revisar
3. **Checkpoints**: `checkpoints/` — versões intermediárias
4. **Relatórios do treino**: loss/perplexity impressos

## 🚀 Próximos passos

1. **Baixar modelo pro seu PC**:
   - Pasta `biomistral-medquad-lora/` no Drive
   - Copiar pra `C:\\Users\\Teste\\Downloads\\Techchalleng3\\biomistral-medquad-lora\\`
   - Rodar `python src/ui/gradio_app.py` — vai detectar e usar modelo real!

2. **Testar a UI Gradio** (localmente) com modelo real

3. **Deploy em HuggingFace Spaces** (opcional, URL pública)

4. **Gravar vídeo demo** (≤15min) pro Tech Challenge

## 📚 Documentação

- **DOCX do projeto**: `TECHCHALLENGE_FASE3_PROJETO_COMPLETO.docx` no repo
- **Manual UI**: `MANUAL_UI.md` no repo
- **Relatório técnico**: `RELATORIO_TECNICO_PARA_EQUIPE.md` no repo